## Curso: MACHINE LEARNING CLASIFICACIÓN Y REGRESIÓN

## Módulo 2: Métodos de validación de información
## Unidad 1: Metaclasificadores

🧩 Objetivo

- Entrenar distintos tipos de ensamble.
- Evaluar desempeños.

## Importación de librerias

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.datasets import fetch_california_housing

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, root_mean_squared_error

## **AdaBoostClassifier**

AdaBoost, un método adaptativo que combina varios clasificadores débiles,
en este caso árboles de decisión pequeños, para crear un clasificador fuerte.

**Dataset**: Iris

Fue introducido en 1936 por el estadístico y biólogo Ronald A. Fisher en un estudio sobre la diferenciación de especies de flores.

Incluye 150 observaciones de flores del género Iris, divididas en 3 especies:

Iris setosa
Iris versicolor
Iris virginica

Para cada flor se registran 4 características numéricas:

Largo del sépalo (sepal length)
Ancho del sépalo (sepal width)
Largo del pétalo (petal length)
Ancho del pétalo (petal width)

Todas las medidas están en centímetros.

El objetivo típico es predecir la especie de la flor a partir de esas cuatro medidas.

En términos de machine learning:

Variables predictoras (features): medidas del sépalo y pétalo

Variable objetivo (target): especie de Iris

### Carga de datos

In [2]:
iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["species"] = iris.target

In [3]:
df.shape

(150, 5)

In [4]:
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   species            150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


### Split

Separamos los datos para entrenar el modelo con una parte (80%) y evaluar su rendimiento con datos que nunca ha visto (20%).

In [6]:
X = df.drop("species", axis=1)
y = df["species"]

In [7]:
# `random_state` se usa para que la división sea siempre la misma y los resultados sean reproducibles.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Modelado

In [8]:
# Instanciar el clasificador base
# Al poner `max_depth=1 es un árbol muy simple
estimator = DecisionTreeClassifier(max_depth=1)

In [9]:
# Instanciar el metaclasificador AdaBoost
# n_estimators: Es el número de clasificadores débiles que se crearán secuencialmente.
# base_estimator: El modelo que se usará en cada iteración.
# algorithm='SAMME.R': Es una versión del algoritmo AdaBoost que suele converger más rápido.
clf = AdaBoostClassifier(
    estimator=estimator,
    n_estimators=100,
    algorithm='SAMME',
    random_state=42
)

In [10]:
# Entrenar el clasificador
clf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=1),
                   n_estimators=100, random_state=42)

In [11]:
# Usamos el modelo entrenado para predecir la especie de las flores en el conjunto de prueba.
y_pred = clf.predict(X_test)

In [12]:
# evaluamos
accuracy = accuracy_score(y_test, y_pred)

accuracy

0.9333333333333333

El dataset Iris es un problema relativamente sencillo, por lo que es común obtener precisiones muy altas.

## Gradient Boosting

Usaremos Gradient Boosting, que construye árboles secuencialmente, donde cada árbol nuevo intenta corregir los errores del anterior.

El objetivo es predecir el valor mediano de las viviendas en California.

**Dataset**:
Datos de inmuebles de california.


### Carga de datos

In [16]:
housing = fetch_california_housing()
X, y = housing.data, housing.target

### Split de datos

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

### Modelado

In [18]:
# n_estimators: Número de árboles a construir.
# max_depth: Profundidad máxima de cada árbol. Controla la complejidad del modelo.
# subsample: Fracción de muestras a usar para entrenar cada árbol. Ayuda a prevenir el sobreajuste.
# learning_rate: tasa de aprendizaje
gbr = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=4,
    subsample=0.8,
    learning_rate=0.1,
    random_state=42
)


In [19]:
# Entrenamos el modelo con los datos de entrenamiento
gbr.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=4, random_state=42, subsample=0.8)

In [21]:
y_pred = gbr.predict(X_test)

In [22]:
# Para regresión, una métrica común es el Error Cuadrático Medio (MSE).
# Mide el promedio de los errores al cuadrado. Cuanto más bajo, mejor.
# También calculamos el R-cuadrado (score), que indica qué proporción de la varianza
# de la variable dependiente es predecible a partir de las variables independientes.
# Un valor cercano a 1 es ideal.
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2_score = gbr.score(X_test, y_test)

print(f"Error Cuadrático Medio (MSE): {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Puntaje R-cuadrado (R²): {r2_score:.4f}")

Error Cuadrático Medio (MSE): 0.2641
RMSE: 0.5139
Puntaje R-cuadrado (R²): 0.7985


## Bagging

Volvemos a Iris...

In [23]:
iris = load_iris()

df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

df["species"] = iris.target

X = df.drop("species", axis=1)
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Modelado

In [24]:
# max_samples: La proporción de muestras del set de entrenamiento que se usará para entrenar cada árbol.
# max_features: La proporción de características que se considerará en cada árbol.
# bootstrap=True: Indica que se usará bootstrapping para crear los subconjuntos de datos.
# n_jobs=-1: Utiliza todos los procesadores disponibles para entrenar los árboles en paralelo. Acelera el proceso.

bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100, # Usamos 100 estimadores en lugar de 10 para un modelo más robusto
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

In [25]:
bagging_clf.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_features=0.8,
                  max_samples=0.8, n_estimators=100, n_jobs=-1,
                  random_state=42)

In [26]:
accuracy = bagging_clf.score(X_test, y_test)

print(f"\nLa precisión (Accuracy) del clasificador Bagging es: {accuracy:.4f}")


La precisión (Accuracy) del clasificador Bagging es: 1.0000


## Random Forest

Random Forest es un ensamble paralelo basado en árboles de decisión.

Utiliza:

- bootstrapping
- selección aleatoria de variables

para generar múltiples árboles y combinar sus predicciones.

In [28]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)



In [29]:
acc_rf = accuracy_score(y_test, pred_rf)
acc_rf

1.0

## XGBoost

XGBoost es una implementación optimizada de **Gradient Boosting**.

Sus principales características:

- regularización para evitar overfitting
- alta eficiencia computacional
- excelente desempeño en problemas tabulares

In [30]:
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:50:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [31]:
acc_xgb = accuracy_score(y_test, pred_xgb)
acc_xgb

1.0

## LightGBM

LightGBM es otro algoritmo de boosting optimizado.

Sus ventajas:

- muy rápido en datasets grandes
- menor consumo de memoria
- crecimiento de árboles más eficiente

In [32]:
lgbm = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

lgbm.fit(X_train, y_train)

pred_lgbm = lgbm.predict(X_test)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000313 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 91
[LightGBM] [Info] Number of data points in the train set: 120, number of used features: 4
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.073920
[LightGBM] [Info] Start training from score -1.123930
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

In [33]:
acc_lgbm = accuracy_score(y_test, pred_lgbm)

acc_lgbm

1.0